# Otomasi Origination **Syariah** sampai *Completed* — NEW LOS

Notebook ini membuat **satu Origination dari nol sampai berstatus Completed** untuk skenario
pembiayaan **Syariah**, lewat 12 fase dan 7 akun. Isinya adalah port dari
`otomasi-origination/buat_origination_syariah.py` (+ `buat_origination.py` dan `los.py`)
ke format Google Colab / Jupyter, dengan Chrome berjalan *headless*.

| Fase | Akun | Langkah |
| --- | --- | --- |
| 1 | Staff Bisnis | Customer baru → pipeline → **fasilitas Syariah** (Segmen Syariah, Skema, Tujuan, sub-form skema) |
| 2 | Staff Bisnis | Submit pipeline + komentar |
| 3 | TL Bisnis | Approve pipeline |
| 4 | Staff Bisnis | Go To Origination (staff DEPI & DH) |
| 5 | Staff Bisnis | Pembahasan Origination: Tanggal Entry, dokumen wajib, Other Document → Submit |
| 6 | TL Bisnis | Submit → Send |
| 7 | Kadiv Bisnis | Approve & Kirim ke LPS |
| 8 | TL DEPI | Isi dokumen wajib DEPI → Submit → Send |
| 9 | Kadiv DEPI | Approve |
| 10 | TL DH | Isi dokumen wajib DH → Submit → Send |
| 11 | Kadiv DH | Approve |
| 12 | Staff Bisnis | Cek status Completed |

> ⚠️ **Membuat data sungguhan** di `BASE_URL`. Pastikan itu lingkungan uji (dev/SIT).
>
> ⚠️ **Jaringan:** runtime Colab berjalan di server Google. Bila server LOS hanya bisa diakses
> dari jaringan internal/VPN, gunakan **Connect → Connect to a local runtime** (Jupyter lokal),
> atau jalankan notebook ini di Jupyter lokal.
>
> ⚠️ `base_url`, username akun, dan data isian dibaca dari **`config.json`** yang tidak ikut di
> repo (formatnya: `config.example.json`). Di Colab, berkas ini diminta untuk diunggah di sel
> Konfigurasi. Jangan membagikan `config.json` secara publik.

Cara pakai: jalankan sel dari atas ke bawah (**Runtime → Run all**). Bila gagal di tengah,
perbaiki penyebabnya, centang `LANJUT`, lalu jalankan ulang sel **Konfigurasi** dan **Jalankan**.

## 1. Instalasi Google Chrome & Selenium

In [ ]:
import os, shutil

DI_COLAB = 'google.colab' in str(get_ipython())

if DI_COLAB and not shutil.which('google-chrome'):
    !wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
    !apt-get update -qq > /dev/null
    !apt-get install -y -qq /tmp/chrome.deb > /dev/null
%pip install -q "selenium>=4.20"

# Chromedriver diunduh otomatis oleh Selenium Manager saat browser pertama kali dibuka.
print('Chrome :', shutil.which('google-chrome') or shutil.which('chrome') or '(pakai Chrome sistem)')

## 2. Konfigurasi

Semua isian berasal dari `config.json` (format sama dengan versi skrip lokal). Skema syariah
diatur di `data.syariah.skema`. Nilai `*_opsi`, `tipe_proses`, dan semua kunci di `syariah` (kecuali teks bebas) adalah **regex**,
karena label opsi berasal dari tabel parameter DB dan bisa berbeda antar lingkungan.

Sub-form skema syariah (Musyarakah: Syirkah SMI + Syirkah Nasabah; Murabahah: Objek + Harga Beli;
Qardh: Jumlah pinjaman; dst.) diisi **generik**: setiap field bertanda `*` yang masih kosong diisi
nominal acak bila punya pilihan mata uang (IDR/USD), selain itu diisi `teks_objek`.

In [ ]:
import json
from pathlib import Path

#@markdown ### Opsi run
LANJUT = False  #@param {type:"boolean"}
SAMPAI = 12  #@param {type:"slider", min:1, max:12, step:1}

FOLDER = Path('/content/otomasi-origination') if DI_COLAB else Path.cwd()
FOLDER.mkdir(parents=True, exist_ok=True)
ARTEFAK = FOLDER / 'artefak'
BERKAS_JEJAK = FOLDER / 'jejak_syariah.json'
BERKAS_PDF = str(FOLDER / 'dokumen.pdf')
BERKAS_KONFIG = FOLDER / 'config.json'

# config.json (base_url, username akun, data isian) TIDAK ada di repo; formatnya lihat
# config.example.json. Di Colab, unggah berkasnya saat diminta.
if not BERKAS_KONFIG.exists() and DI_COLAB:
    from google.colab import files
    print('Unggah config.json:')
    unggahan = files.upload()
    BERKAS_KONFIG.write_bytes(unggahan[next(n for n in unggahan if n.endswith('.json'))])
assert BERKAS_KONFIG.exists(), f'{BERKAS_KONFIG} tidak ada. Salin config.example.json lalu isi.'

KONFIG = json.loads(BERKAS_KONFIG.read_text(encoding='utf-8'))
BASE_URL = KONFIG['base_url']
AKUN = KONFIG['akun']
DATA = KONFIG['data']
SYARIAH = DATA['syariah']

kosong = [k for k, v in AKUN.items() if not str(v).strip()]
assert BASE_URL.strip(), 'base_url di config.json masih kosong.'
assert not kosong, f'Username akun di config.json belum diisi: {", ".join(kosong)}'
print('Folder kerja:', FOLDER)
print('Skema syariah:', SYARIAH['skema'])

## 3. Dokumen PDF untuk diunggah

Semua dokumen wajib dan Other Document memakai satu berkas PDF. Sel ini membuat PDF minimal
secara otomatis. Bila ingin memakai PDF sendiri, set `UNGGAH_SENDIRI = True`.

In [ ]:
UNGGAH_SENDIRI = False  #@param {type:"boolean"}


def buat_pdf_minimal(path: str, teks: str = 'Dokumen uji otomasi Origination Syariah'):
    """PDF satu halaman yang valid (xref dihitung), tanpa library tambahan."""
    isi = f'BT /F1 18 Tf 72 720 Td ({teks}) Tj ET'.encode('latin-1')
    objek = [
        b'<< /Type /Catalog /Pages 2 0 R >>',
        b'<< /Type /Pages /Kids [3 0 R] /Count 1 >>',
        b'<< /Type /Page /Parent 2 0 R /MediaBox [0 0 595 842] /Contents 4 0 R'
        b' /Resources << /Font << /F1 5 0 R >> >> >>',
        b'<< /Length %d >>\nstream\n' % len(isi) + isi + b'\nendstream',
        b'<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>',
    ]
    out = bytearray(b'%PDF-1.4\n')
    posisi = []
    for i, o in enumerate(objek, start=1):
        posisi.append(len(out))
        out += b'%d 0 obj\n' % i + o + b'\nendobj\n'
    awal_xref = len(out)
    out += b'xref\n0 %d\n0000000000 65535 f \n' % (len(objek) + 1)
    out += b''.join(b'%010d 00000 n \n' % p for p in posisi)
    out += b'trailer\n<< /Size %d /Root 1 0 R >>\nstartxref\n%d\n%%%%EOF\n' % (len(objek) + 1, awal_xref)
    Path(path).write_bytes(bytes(out))


if UNGGAH_SENDIRI and DI_COLAB:
    from google.colab import files
    unggahan = files.upload()
    nama = next(n for n in unggahan if n.lower().endswith('.pdf'))
    Path(BERKAS_PDF).write_bytes(unggahan[nama])
elif not Path(BERKAS_PDF).exists():
    buat_pdf_minimal(BERKAS_PDF)
print(BERKAS_PDF, Path(BERKAS_PDF).stat().st_size, 'byte')

## 4. Helper Selenium (`los.py`)

Catatan DOM aplikasi NEW LOS (Next.js + MUI):
- Loader global adalah `MuiDialog` berisi spinner saja, jadi popup selalu dicari lewat **teksnya**.
- Popup Success = "Success" + "...berhasil..." + Close; Error = "Error!" + Close.
- MUI Select dirender sebagai `[role=combobox]`; opsinya `[role=option]` di listbox portal (luar form).

Perbedaan dari versi lokal: Chrome selalu headless dengan `--no-sandbox` dan
`--disable-dev-shm-usage` (wajib di Colab).

In [ ]:
import re
import time
from datetime import date

from selenium import webdriver
from selenium.common.exceptions import (
    ElementClickInterceptedException,
    StaleElementReferenceException,
    WebDriverException,
)
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.remote.webelement import WebElement


def log(pesan: str):
    print(f'[{time.strftime("%H:%M:%S")}] {pesan}', flush=True)


def hari_ini() -> str:
    return date.today().strftime('%d/%m/%Y')


class GagalLangkah(Exception):
    """Kegagalan yang pesannya sudah cukup jelas untuk dibaca manusia."""


def _teks(el: WebElement) -> str:
    try:
        return re.sub(r'\s+', ' ', el.text or '').strip()
    except StaleElementReferenceException:
        return ''


def _tampil(el: WebElement) -> bool:
    try:
        return el.is_displayed()
    except StaleElementReferenceException:
        return False


def xp_literal(s: str) -> str:
    """String aman untuk XPath (menangani tanda kutip)."""
    if "'" not in s:
        return f"'{s}'"
    if '"' not in s:
        return f'"{s}"'
    return "concat('" + s.replace("'", "',\"'\",'") + "')"


class Los:
    def __init__(self, base_url: str):
        self.base_url = base_url.rstrip('/')
        opsi = webdriver.ChromeOptions()
        opsi.add_argument('--headless=new')
        opsi.add_argument('--no-sandbox')
        opsi.add_argument('--disable-dev-shm-usage')
        opsi.add_argument('--window-size=1440,900')
        opsi.add_argument('--ignore-certificate-errors')
        opsi.add_experimental_option('excludeSwitches', ['enable-logging'])
        self.d = webdriver.Chrome(options=opsi)
        self.d.implicitly_wait(0)

    # ------------------------------------------------------------------ dasar

    def tutup(self):
        try:
            self.d.quit()
        except WebDriverException:
            pass

    def simpan_bukti(self, nama: str):
        """Screenshot + HTML saat gagal, untuk dilihat sesudahnya."""
        ARTEFAK.mkdir(exist_ok=True)
        cap = time.strftime('%Y%m%d-%H%M%S')
        png = ARTEFAK / f'{cap}-{nama}.png'
        try:
            self.d.save_screenshot(str(png))
            (ARTEFAK / f'{cap}-{nama}.html').write_text(self.d.page_source, encoding='utf-8')
        except WebDriverException:
            pass
        return png

    def tunggu(self, fungsi, batas: float = 60, pesan: str = 'kondisi tidak terpenuhi', jeda: float = 0.5):
        """Ulangi `fungsi` sampai mengembalikan nilai truthy, atau gagal setelah `batas` detik."""
        akhir = time.time() + batas
        while True:
            try:
                hasil = fungsi()
                if hasil:
                    return hasil
            except (StaleElementReferenceException, ElementClickInterceptedException):
                pass
            if time.time() > akhir:
                raise GagalLangkah(f'{pesan} (setelah {int(batas)} detik)')
            time.sleep(jeda)

    def cari(self, xpath: str, induk=None) -> list[WebElement]:
        return [e for e in (induk or self.d).find_elements(By.XPATH, xpath) if _tampil(e)]

    def satu(self, xpath: str, batas: float = 60, induk=None, pesan: str = None) -> WebElement:
        return self.tunggu(lambda: (self.cari(xpath, induk) or [None])[0],
                           batas, pesan or f'elemen tidak muncul: {xpath}')

    def klik(self, el: WebElement):
        self.d.execute_script('arguments[0].scrollIntoView({block:"center"})', el)
        try:
            el.click()
        except (ElementClickInterceptedException, WebDriverException):
            self.d.execute_script('arguments[0].click()', el)

    def klik_xpath(self, xpath: str, batas: float = 60, induk=None, pesan: str = None):
        el = self.tunggu(lambda: next((e for e in self.cari(xpath, induk) if e.is_enabled()), None),
                         batas, pesan or f'tombol tidak muncul/aktif: {xpath}')
        self.klik(el)
        return el

    def ketik(self, el: WebElement, teks: str):
        self.klik(el)
        el.send_keys(Keys.CONTROL, 'a')
        el.send_keys(Keys.DELETE)
        el.send_keys(teks)

    def tombol(self, teks: str, induk=None) -> str:
        """XPath tombol dengan teks persis."""
        return f".//button[normalize-space()={xp_literal(teks)}]" if induk else f"//button[normalize-space()={xp_literal(teks)}]"

    def buka(self, path: str):
        self.d.get(self.base_url + path)
        time.sleep(1)
        if '/login' in self.d.current_url and not path.startswith('/login'):
            raise GagalLangkah('Terlempar ke halaman login — sesi habis?')
        self.buffering()

    # ------------------------------------------------------------------ menunggu

    # Hanya spinner yang BERPUTAR (indeterminate). Progress bar determinate dipakai sebagai
    # indikator kelengkapan stepper ("Validasi 0%") dan tampil terus — bukan tanda loading.
    _JS_SIBUK = """
      const tampil = e => !!(e.offsetWidth || e.offsetHeight || e.getClientRects().length);
      const spinner = [...document.querySelectorAll(
        '.MuiCircularProgress-indeterminate, .MuiLinearProgress-indeterminate, .MuiLinearProgress-query')].some(tampil);
      const teks = /Auto Saving\\.\\.\\.|Sedang Submit Dokumen|Mengunggah/.test(document.body ? document.body.innerText : '');
      return spinner || teks;
    """

    def buffering(self, batas: float = 120):
        """Tunggu sampai spinner/loader hilang. Lunak: lewat batas hanya dicatat, tidak gagal."""
        time.sleep(0.6)
        akhir = time.time() + batas
        while time.time() < akhir:
            try:
                if not self.d.execute_script(self._JS_SIBUK):
                    time.sleep(0.4)
                    if not self.d.execute_script(self._JS_SIBUK):
                        return
            except WebDriverException:
                pass
            time.sleep(0.5)
        log(f'  (masih ada indikator loading setelah {int(batas)} detik, lanjut)')

    # ------------------------------------------------------------------ popup

    def popup(self, pola: str) -> WebElement | None:
        """Popup MUI yang tampil dan teksnya cocok dengan regex `pola` (paling atas bila bertumpuk)."""
        cocok = [p for p in self.d.find_elements(By.CSS_SELECTOR, '.MuiDialog-root')
                 if _tampil(p) and re.search(pola, _teks(p), re.I)]
        return cocok[-1] if cocok else None

    def tunggu_popup(self, pola: str, batas: float = 60) -> WebElement:
        return self.tunggu(lambda: self.popup(pola), batas, f'popup "{pola}" tidak muncul')

    def sukses_lalu_tutup(self, pola: str = r'berhasil', batas: float = 180) -> str:
        """
        Tunggu popup sukses lalu klik Close. Popup Error langsung menggagalkan run dengan
        isi pesannya — tanpa ini kegagalan backend hanya terlihat sebagai timeout.
        """
        def cek():
            ok = self.popup(pola)
            if ok:
                return ok
            err = self.popup(r'Error!|Terjadi kesalahan')
            if err:
                raise GagalLangkah(f'Aplikasi menampilkan error: {_teks(err)}')
            return None

        p = self.tunggu(cek, batas, f'popup sukses "{pola}" tidak muncul')
        teks = _teks(p)
        self.klik_xpath(self.tombol('Close', p), 15, induk=p)
        self.tunggu(lambda: not _tampil(p), 20, 'popup sukses tidak tertutup')
        self.buffering()
        return teks

    def konfirmasi_bila_ada(self, pola: str, teks_tombol: str, batas: float = 5) -> bool:
        """Klik tombol di popup konfirmasi bila popup itu muncul dalam `batas` detik."""
        try:
            p = self.tunggu_popup(pola, batas)
        except GagalLangkah:
            return False
        self.klik_xpath(self.tombol(teks_tombol, p), 15, induk=p)
        self.buffering()
        return True

    # ------------------------------------------------------------------ input

    def opsi(self, pola: str) -> list[WebElement]:
        return [o for o in self.cari("//*[@role='option']") if re.search(pola, _teks(o), re.I)]

    def klik_opsi(self, pola: str, pesan: str) -> str:
        """
        Cari opsi yang cocok lalu klik, dalam SATU percobaan `tunggu`. Daftar opsi autocomplete
        berbasis API bisa dirender ulang di antara "ditemukan" dan "diklik" (hasil pencarian tiba
        dua kali), sehingga elemennya basi; percobaan berikutnya mencari ulang dari awal.
        Mengembalikan label opsi yang diklik.
        """
        def cari_lalu_klik():
            o = next((x for x in self.opsi(pola) if _teks(x)), None)
            if o is None:
                return None
            label = _teks(o)
            self.klik(o)
            return label

        return self.tunggu(cari_lalu_klik, 30, pesan)

    def pilih_select(self, teks_sekarang: str, pola_opsi: str, induk=None):
        """MUI Select: dikenali dari teks yang sedang tampil (biasanya placeholder)."""
        kotak = self.satu(f".//*[@role='combobox'][contains(normalize-space(), {xp_literal(teks_sekarang)})]",
                          30, induk or self.d, f'dropdown "{teks_sekarang}" tidak ditemukan')
        self.klik(kotak)
        self.klik_opsi(pola_opsi, f'opsi "{pola_opsi}" tidak ada di dropdown "{teks_sekarang}"')
        self.tunggu(lambda: not self.cari("//*[@role='listbox']"), 10, 'dropdown tidak tertutup')
        time.sleep(0.3)

    def pilih_select_berlabel(self, label: str, pola_opsi: str, induk=None):
        """MUI Select yang sudah berisi nilai (placeholder tak terlihat): dicari lewat teks labelnya."""
        kotak = self.satu(f".//*[normalize-space(text())={xp_literal(label)}]"
                          f"/ancestor::div[.//*[@role='combobox']][1]//*[@role='combobox']",
                          30, induk or self.d, f'dropdown berlabel "{label}" tidak ditemukan')
        self.klik(kotak)
        self.klik_opsi(pola_opsi, f'opsi "{pola_opsi}" tidak ada di dropdown "{label}"')
        self.tunggu(lambda: not self.cari("//*[@role='listbox']"), 10, 'dropdown tidak tertutup')
        time.sleep(0.5)

    def pilih_autocomplete(self, kotak: WebElement, kata: str, pola_opsi: str):
        """MUI Autocomplete berbasis API: ketik kata kunci, tunggu opsi yang cocok, pilih."""
        self.ketik(kotak, kata)
        self.klik_opsi(pola_opsi, f'opsi "{pola_opsi}" tidak muncul setelah mengetik "{kata}"')
        time.sleep(0.5)

    def pilih_dari_daftar(self, kotak: WebElement, pola_opsi: str = '') -> str:
        """
        MUI Autocomplete yang daftarnya muncul saat diklik (tanpa mengetik). Pilih opsi yang
        cocok dengan `pola_opsi`, atau opsi pertama bila pola kosong. Mengembalikan labelnya.
        """
        self.tunggu(lambda: kotak.is_enabled(), 30, 'dropdown masih nonaktif')
        self.klik(kotak)
        label = self.klik_opsi(pola_opsi or '.', f'opsi "{pola_opsi or "(pertama)"}" tidak muncul')
        time.sleep(0.5)
        return label

    def pilih_radio(self, nama: str, nilai: str):
        r = self.d.find_element(By.CSS_SELECTOR, f'input[type="radio"][name="{nama}"][value="{nilai}"]')
        self.d.execute_script('arguments[0].click()', r)
        self.tunggu(lambda: r.is_selected(), 5, f'radio {nama}={nilai} tidak terpilih')

    def ketik_tanggal(self, kotak: WebElement, tanggal: str, xpath_netral: str):
        """
        Date picker MUI (DD/MM/YYYY): diketik digit per digit, lalu kalendernya ditutup dengan
        mengklik elemen netral (judul halaman). Sengaja bukan Escape: Escape ikut menutup popup lain.
        """
        self.klik(kotak)
        kotak.send_keys(Keys.CONTROL, 'a')
        kotak.send_keys(tanggal.replace('/', ''))
        time.sleep(0.3)
        netral = self.cari(xpath_netral)
        if netral:
            self.klik(netral[0])
        time.sleep(0.3)

    # ------------------------------------------------------------------ sesi

    def login(self, username: str):
        log(f'Login sebagai {username}')
        self.d.delete_all_cookies()
        self.d.get(self.base_url + '/login')
        kotak = self.satu("//input[@id='input-username']", 60)
        self.ketik(kotak, username)
        self.klik_xpath("//*[@id='login-button']")
        try:
            self.tunggu(lambda: '/login' not in self.d.current_url, 60, 'masih di halaman login')
        except GagalLangkah:
            pesan = ''
            try:
                alert = self.d.switch_to.alert
                pesan = alert.text
                alert.dismiss()
            except WebDriverException:
                pass
            raise GagalLangkah(f'Login {username} gagal. {pesan or "Mungkin butuh sandi/OTP."}')
        self.buffering()

    def logout(self):
        """Tombol header "Logout" → popup "Apakah anda yakin ingin keluar?" → Ya."""
        try:
            tombol = self.cari(self.tombol('Logout'))
            if tombol:
                self.klik(tombol[0])
                self.konfirmasi_bila_ada(r'yakin ingin keluar', 'Ya', 10)
                self.tunggu(lambda: '/login' in self.d.current_url, 30, 'tidak kembali ke halaman login')
        except GagalLangkah:
            pass
        # Jaring pengaman: sesi berikutnya harus mulai bersih apa pun yang terjadi di atas.
        self.d.delete_all_cookies()
        log('Logout')

    # ------------------------------------------------------------------ To Do List

    def buka_tugas(self, customer: str, url_cadangan: str, batas: float = 150) -> bool:
        """
        Dashboard → To Do List / Approval List → Filter "Nama Customer" → Apply → "Go to task".
        Bila tugas belum muncul setelah `batas` detik, buka `url_cadangan` langsung.
        """
        baris_xp = (f"//button[normalize-space()='Go to task']"
                    f"/parent::*[*[normalize-space()={xp_literal(customer)}]]")
        akhir = time.time() + batas
        while time.time() < akhir:
            self.buka('/')
            self.satu("//*[normalize-space(text())='To Do List' or normalize-space(text())='Approval List']", 60)
            # Tombol Filter pertama milik To Do List (yang kedua milik Inquiry Data).
            self.klik_xpath("(//button[normalize-space()='Filter'])[1]", 30)
            pop = self.satu("//*[contains(@class,'MuiPopover-root')]", 15)
            self.ketik(self.satu(".//input[@placeholder='example: proses..']", 15, pop), customer)
            self.klik_xpath(self.tombol('Apply', pop), 15, induk=pop)
            self.buffering()
            time.sleep(1.5)
            baris = self.cari(baris_xp)
            if baris:
                self.klik(baris[0].find_element(By.XPATH, ".//button[normalize-space()='Go to task']"))
                self.tunggu(lambda: self.d.current_url.rstrip('/') != self.base_url, 60, 'Go to task tidak berpindah halaman')
                self.buffering()
                log(f'  Go to task → {self.d.current_url.replace(self.base_url, "")}')
                return True
            log('  tugas belum muncul di dashboard, coba lagi 15 detik lagi')
            time.sleep(15)
        log(f'  tugas tidak ditemukan di dashboard, buka langsung {url_cadangan}')
        self.buka(url_cadangan)
        return False

## 5. Rute, jejak run, dan data uji acak

In [ ]:
import json
import random

RUTE_PIPELINE = '/loan-processing/submission-proposal/pipeline'
RUTE_ORIGINATION = '/origination-submission/origination'

PERUSAHAAN = [
    'Garuda Nusantara', 'Telkom Sinergi', 'Pertamina Energi', 'Astra Mandiri', 'Indofood Prima',
    'Unilever Nusa', 'Semen Gresik Raya', 'Wijaya Karya Infra', 'Jasa Marga Tol', 'Bukit Asam Daya',
    'Angkasa Pura Logistik', 'Pelindo Samudra', 'Waskita Beton', 'Adhi Karya Persada', 'Hutama Karya Jaya',
]


def detail_pipeline(pid: str) -> str:
    return f'{RUTE_PIPELINE}/{pid}/detail'


def pembahasan(fas: str) -> str:
    return f'{RUTE_ORIGINATION}/{fas}/origination-creation'


def eksak(teks: str) -> str:
    """Regex cocok-persis (case-insensitive) untuk label opsi."""
    return rf'^\s*{re.escape(teks)}\s*$'


# Fase yang selesai dicatat di jejak; LANJUT=True melewatinya, jadi kegagalan di fase Kadiv DH
# tidak memaksa membuat customer baru dari awal.
def baca_jejak() -> dict:
    return json.loads(BERKAS_JEJAK.read_text(encoding='utf-8')) if BERKAS_JEJAK.exists() else {}


def simpan_jejak(**isi):
    j = baca_jejak()
    j.update(isi)
    BERKAS_JEJAK.write_text(json.dumps(j, indent=1, ensure_ascii=False), encoding='utf-8')


def nama_customer_baru() -> str:
    """
    Huruf pertama kapital, TIDAK diawali PT/TBK/dll. dan tidak diakhiri Persero/TBK.
    Akhiran unik supaya barisnya bisa dicari di To Do List / Origination List.
    """
    kode = ''.join(random.choices('ABCDEFGHJKLMNPQRSTUVWXYZ', k=4))
    return f'{random.choice(PERUSAHAAN)} Oto {kode}'


def nominal_acak() -> str:
    """1–99 milyar dengan sisa jutaan supaya tidak selalu bulat."""
    return str(random.randint(1, 99) * 1_000_000_000 + random.randint(0, 899) * 1_000_000)


def nomor_dokumen_acak() -> str:
    return str(random.randint(100_000, 999_999))


def rupiah(angka: str) -> str:
    return 'Rp' + f'{int(angka):,}'.replace(',', '.')

## 6. Form fasilitas **Syariah**

Begitu Segmen Pembiayaan = Syariah, "Produk" berganti label menjadi **Skema pembiayaan**
(opsi dari parameter `productSyariah`) dan "Nominal Pengajuan" hilang, diganti sub-form per skema
(`ModalFormFacility.tsx`, `SyariahForm.tsx`). Label sub-form bisa diganti dari konfigurasi skema
di DB, jadi diisi generik lewat `isi_field_wajib_kosong`.

In [ ]:
def isi_fasilitas_syariah(los: Los, form):
    """Segmen Pembiayaan → Syariah, lalu Skema pembiayaan, Tujuan Pembiayaan, dan sub-form skema."""
    # Segmen sudah terisi bawaan divisi, jadi placeholder-nya tak terlihat: dicari lewat label.
    los.pilih_select_berlabel('Segmen Pembiayaan', SYARIAH['segmen'], induk=form)
    los.buffering()
    los.pilih_select('Choose Skema pembiayaan', SYARIAH['skema'], induk=form)
    los.buffering()
    if los.cari(".//*[contains(normalize-space(), 'Skema belum dimapping')]", form):
        raise GagalLangkah(f'Skema "{SYARIAH["skema"]}" belum dimapping di lingkungan ini (pesan aplikasi: '
                           '"Skema belum dimapping, mohon kontak Admin"). Pilih skema lain di Konfigurasi.')
    los.ketik(los.satu(".//input[@placeholder='Tujuan Pembiayaan']", 15, form), SYARIAH['tujuan_pembiayaan'])
    log(f'  fasilitas syariah: skema {SYARIAH["skema"]}')
    isi_field_wajib_kosong(los, form, SYARIAH['teks_objek'])


def isi_field_wajib_kosong(los: Los, form, teks: str):
    """
    Isi setiap field bertanda * di `form` yang masih kosong dan bisa diketik: nominal acak bila
    wadahnya memuat pilihan mata uang (IDR/USD), selain itu `teks`. Dropdown, autocomplete,
    tanggal, dan field nonaktif (mis. "Mitra Syarik SMI", "Total Syirkah") dilewati.
    """
    xp_isian = (".//input[(not(@type) or @type='text' or @type='number')"
                " and not(@disabled) and not(@readonly) and not(@aria-hidden='true')"
                " and not(@role='combobox') and not(contains(@placeholder, 'DD/MM'))"
                " and not(contains(@class, 'MuiSelect-nativeInput'))] | .//textarea[not(@disabled)]")
    for bintang in los.cari(".//*[normalize-space(text())='*']", form):
        label = bintang.find_element(By.XPATH, '..')
        nama = re.sub(r'\s*\*\s*$', '', re.sub(r'\s+', ' ', label.text)).strip()
        wadah = label.find_element(By.XPATH, './ancestor::div[.//input or .//textarea][1]')
        isian = [i for i in los.cari(xp_isian, wadah) if not i.get_attribute('value')]
        if not isian:
            continue
        mata_uang = any(re.search(r'^(IDR|USD)$', c.text.strip())
                        for c in wadah.find_elements(By.XPATH, ".//*[@role='combobox']"))
        nilai = nominal_acak() if mata_uang else teks
        los.ketik(isian[0], nilai)
        log(f'    {nama}: {rupiah(nilai) if mata_uang else nilai}')
        los.buffering(10)

## 7. Fase 1–4: pipeline, submit, approve TL, Go To Origination

In [ ]:
def fase_buat_pipeline(los: Los):
    customer = nama_customer_baru()
    log(f'  customer: {customer}')

    # Loan Processing → Proposal Pengajuan → Pipeline → Add New Pipeline
    los.buka(RUTE_PIPELINE)
    los.klik_xpath("//button[contains(normalize-space(), 'Add New Pipeline')]")
    cari = los.satu("//div[contains(@class,'MuiDialog-root')]//input[@placeholder='Pencarian...']", 30)
    los.ketik(cari, DATA['kata_pencarian'])

    # Tunggu 3 detik → Create New Customer & Pipeline → Ya
    time.sleep(3)
    los.buffering()
    los.klik_xpath("//button[normalize-space()='Create New Customer & Pipeline']", 60)

    # Popup "Customer sudah ada dalam pengajuan" hanya muncul bila pencarian ada hasilnya, dan
    # bisa terlambat saat server lambat — jadi tunggu mana yang lebih dulu: pindah halaman atau popup.
    def lanjut_ke_create():
        if 'create-new-debtor' in los.d.current_url:
            return True
        p = los.popup(r'Customer sudah ada dalam pengajuan')
        ya = los.cari(los.tombol('Ya', p), p) if p else []
        if ya:
            los.klik(ya[0])
            time.sleep(1.5)
        return False

    los.tunggu(lanjut_ke_create, 90, 'halaman Create Pipeline tidak terbuka', jeda=1)
    los.buffering()

    # Institution Type, Nama Customer, General Account Manager
    los.pilih_select('Institution Type', eksak(DATA['institution_type']))
    los.ketik(los.satu("//input[@placeholder='Nama Customer']"), customer)
    los.pilih_autocomplete(los.satu("//input[@placeholder='Input General Account Manager']"),
                           DATA['gam_cari'], DATA['gam_opsi'])

    # Save → (peringatan DK / rekomendasi nama) → Success → Close → halaman detail pipeline
    los.klik_xpath("//*[@id='button-save']", 30, pesan='tombol Save belum aktif — ada field wajib yang kosong?')
    los.buffering()
    los.konfirmasi_bila_ada(r'Terdapat kemiripan', 'Save', 3)
    if los.konfirmasi_bila_ada(r'Rekomendasi Nama Customer', 'Create New Customer & Pipeline', 3):
        los.konfirmasi_bila_ada(r'serupa pada rekomendasi', 'Ya', 8)
    los.sukses_lalu_tutup()
    los.tunggu(lambda: re.search(r'/pipeline/[^/]+/detail', los.d.current_url), 60, 'tidak pindah ke detail pipeline')
    pipeline_id = re.search(r'/pipeline/([^/]+)/detail', los.d.current_url).group(1)
    simpan_jejak(customer=customer, pipeline_id=pipeline_id)
    log(f'  pipeline: {pipeline_id}')
    los.buffering()

    # Data pipeline (untuk customer baru, field ini baru bisa diisi di halaman detail)
    los.pilih_select('Jenis Customer', eksak(DATA['jenis_customer']))
    los.pilih_radio('isGroup', 'false')          # Customer Memiliki Group: Tidak
    los.pilih_radio('isRelatedToSmi', 'false')   # Terkait Dengan SMI: Tidak
    los.pilih_select('Tipe Proses', DATA['tipe_proses'])
    los.pilih_select('Tipe Pembiayaan', eksak(DATA['tipe_pembiayaan']))
    los.pilih_autocomplete(los.satu("//input[@placeholder='Pilih PIC2']"), DATA['pic2_cari'], DATA['pic2_opsi'])
    los.pilih_autocomplete(los.satu("//input[@placeholder='Nama Analis']"), DATA['analis_cari'], DATA['analis_opsi'])

    los.klik_xpath("(//button[normalize-space()='Save'])[last()]", 60)
    los.konfirmasi_bila_ada(r'DATA MANDATORY', 'Ya', 3)
    los.sukses_lalu_tutup()

    # Fasilitas Pembiayaan → Add New → Create New → Confirm → isi (SYARIAH) → Save
    los.klik_xpath("//button[normalize-space()='Add New']")
    pemilih = los.tunggu_popup('Tambah Fasilitas Pembiayaan')
    los.d.execute_script('arguments[0].click()',
                         pemilih.find_element(By.CSS_SELECTOR, 'input[type="radio"][value="new"]'))
    los.klik_xpath(los.tombol('Confirm', pemilih), 15, induk=pemilih)
    form = los.tunggu_popup('Add New Fasilitas Pembiayaan', 30)
    los.buffering()
    isi_fasilitas_syariah(los, form)
    los.klik_xpath(los.tombol('Save', form), 15, induk=form)
    los.sukses_lalu_tutup()


def isi_komentar_lalu_simpan(los: Los, komentar: str):
    p = los.tunggu_popup(r'Comment', 30)
    los.ketik(los.satu(".//textarea[@placeholder='Comment']", 15, p), komentar)
    los.klik_xpath(los.tombol('Save', p), 15, induk=p)


def fase_submit_pipeline(los: Los):
    """
    Submit → Comment → Save → Success → Close.
    Server dev kadang menjawab "timeout of 60000ms exceeded" padahal Submit-nya tetap diproses;
    karena itu sebelum tiap percobaan dicek dulu apakah tombol Submit masih ada.
    """
    pid = baca_jejak()['pipeline_id']
    for percobaan in range(1, 4):
        los.buka(detail_pipeline(pid))
        los.satu("//*[normalize-space(text())='Fasilitas Pembiayaan']", 120, pesan='detail pipeline tidak termuat')
        los.buffering()
        if not los.cari("//button[normalize-space()='Submit']"):
            log('  tombol Submit sudah tidak ada — pipeline sudah ter-Submit')
            return
        try:
            los.klik_xpath("(//button[normalize-space()='Submit'])[last()]", 60,
                           pesan='Submit belum aktif — ada field mandatory pipeline yang kosong?')
            los.konfirmasi_bila_ada(r'yakin ingin Submit', 'Ya', 3)
            isi_komentar_lalu_simpan(los, DATA['komentar_submit'])
            los.sukses_lalu_tutup()
            return
        except GagalLangkah as e:
            if 'timeout' not in str(e).lower() or percobaan == 3:
                raise
            log(f'  server timeout saat Submit (percobaan {percobaan}), cek ulang 20 detik lagi')
            time.sleep(20)


def fase_tl_approve_pipeline(los: Los):
    j = baca_jejak()
    los.buka_tugas(j['customer'], detail_pipeline(j['pipeline_id']))
    los.klik_xpath("(//button[normalize-space()='Approve'])[last()]", 60)
    los.konfirmasi_bila_ada(r'yakin ingin Submit', 'Ya', 3)
    isi_komentar_lalu_simpan(los, DATA['komentar'])
    los.sukses_lalu_tutup()


def tunggu_baris_origination(los: Los, customer: str, batas: float):
    xp = f"//tbody/tr[contains(normalize-space(), {xp_literal(customer)})]"
    akhir = time.time() + batas
    while True:
        los.buka(RUTE_ORIGINATION)
        try:
            return los.satu(xp, 20)
        except GagalLangkah:
            if time.time() > akhir:
                raise GagalLangkah(f'{customer} tidak muncul di Origination List setelah {int(batas)} detik')
            time.sleep(10)


def fase_go_to_origination(los: Los):
    j = baca_jejak()
    customer = j['customer']
    los.buka_tugas(customer, detail_pipeline(j['pipeline_id']))
    if '/pipeline/' not in los.d.current_url:
        los.buka(detail_pipeline(j['pipeline_id']))

    los.klik_xpath("//button[contains(normalize-space(), 'Go To Origination')]", 60)
    notif = los.tunggu_popup('Notification to DEPI', 30)
    los.buffering()

    # Dua field "Nama Staff" (multi-select): urutan 1 = DEPI, urutan 2 = DH.
    for urutan, cari, opsi in ((0, DATA['staff_depi_cari'], DATA['staff_depi_opsi']),
                               (1, DATA['staff_dh_cari'], DATA['staff_dh_opsi'])):
        kotak = los.cari(".//div[contains(@class,'MuiAutocomplete-root')]//input", notif)[urutan]
        los.pilih_autocomplete(kotak, cari, opsi)
        los.klik(los.satu(".//*[normalize-space(text())='Selected Pipeline']", 10, notif))

    los.ketik(los.satu(".//textarea[@placeholder='Comment']", 15, notif), DATA['komentar'])
    los.klik_xpath(los.tombol('Save', notif), 15, induk=notif,
                   pesan='Save belum aktif — staff DEPI/DH/komentar belum lengkap?')
    los.sukses_lalu_tutup()
    los.tunggu(lambda: RUTE_ORIGINATION in los.d.current_url, 60, 'tidak pindah ke Origination List')

    # Bucket baru muncul 14–67 detik setelah Go To Origination (diproses asinkron).
    log('  menunggu origination muncul di Origination List...')
    baris = tunggu_baris_origination(los, customer, 300)
    fas_id = baris.find_element(By.XPATH, './td[6]').text.strip()
    simpan_jejak(fas_id=fas_id)
    log(f'  origination: {fas_id}')

## 8. Fase 5–12: Pembahasan Origination, approval berjenjang, cek Completed

In [ ]:
JUDUL_HALAMAN = "//*[normalize-space(text())='Pembahasan Origination']"


def buka_stepper_pembahasan(los: Los):
    if 'origination-creation' not in los.d.current_url:
        los.klik_xpath(f'({JUDUL_HALAMAN})[1]', 60)
        los.tunggu(lambda: 'origination-creation' in los.d.current_url, 60, 'stepper Pembahasan Origination tidak terbuka')
    los.satu("//button[contains(normalize-space(), 'Filter Dokumen')]", 120,
             pesan='halaman Pembahasan Origination tidak selesai dimuat')
    los.buffering()


def isi_tanggal_entry(los: Los):
    kotak = los.cari("//input[@placeholder='Pilih tanggal']")
    if kotak and kotak[0].is_enabled() and not kotak[0].get_attribute('value'):
        log('  isi Tanggal Entry Pengajuan')
        los.ketik_tanggal(kotak[0], hari_ini(), f'({JUDUL_HALAMAN})[last()]')


def isi_dokumen_wajib(los: Los):
    """
    Isi dokumen bertanda * yang terlihat oleh peran ini: Nomor (acak 6 digit), Tanggal
    (hari ini), dan satu PDF. Bila tidak ada satu pun yang bertanda wajib, dokumen pertama
    yang diisi — Submit baru aktif kalau ada minimal satu berkas Draft.
    """
    xp_blok = ("//input[@placeholder='Input Nomor Dokumen']"
               "/ancestor::div[.//button[normalize-space()='Tambah file'] or .//input[@type='file']][1]")

    def judul(blok):
        return re.sub(r'\s+', ' ', blok.text.split('Nomor Dokumen')[0]).strip()

    semua = los.cari(xp_blok)
    if not semua:
        raise GagalLangkah('Tidak ada dokumen yang bisa diisi oleh peran ini di Pembahasan Origination')
    target = [i for i, b in enumerate(semua) if '*' in judul(b)] or [0]

    for i in target:
        blok = los.cari(xp_blok)[i]
        nama = judul(blok)
        log(f'  isi dokumen: {nama}')
        los.ketik(blok.find_element(By.XPATH, ".//input[@placeholder='Input Nomor Dokumen']"), nomor_dokumen_acak())
        los.ketik_tanggal(blok.find_element(By.XPATH, ".//input[@placeholder='DD/MM/YYYY']"), hari_ini(),
                          f'({JUDUL_HALAMAN})[last()]')

        tambah = los.cari(".//button[normalize-space()='Tambah file']", blok)
        if tambah:
            los.klik(tambah[0])
            time.sleep(0.8)
        unggah = los.tunggu(lambda: blok.find_elements(By.XPATH, ".//input[@type='file']"), 15,
                            f'kotak unggah "{nama}" tidak muncul')[0]
        unggah.send_keys(BERKAS_PDF)

        # Unggah tidak memunculkan popup: tunggu kartu berkas berstatus Draft.
        los.tunggu(lambda: los.cari(".//*[normalize-space(text())='Draft']", blok), 120,
                   f'unggah ke "{nama}" tidak menjadi Draft')
        los.buffering()


def tambah_other_document(los: Los):
    """
    Other Document → Add New → Create New → Confirm → "Add Dokumen": Kategori → Group → Jenis
    (berantai), unggah PDF, Nomor (acak), Tanggal (hari ini), Keterangan → Save.
    """
    xp_section = ("//*[normalize-space(text())='Other Document']"
                  "/ancestor::div[.//button[normalize-space()='Add New']][1]")
    los.klik_xpath(f"{xp_section}//button[normalize-space()='Add New']", 30,
                   pesan='tombol Add New di Other Document tidak ada')

    pemilih = los.tunggu_popup(r'Create New', 15)
    los.d.execute_script('arguments[0].click()',
                         pemilih.find_element(By.CSS_SELECTOR, 'input[type="radio"][value="new"]'))
    los.klik_xpath(los.tombol('Confirm', pemilih), 15, induk=pemilih)

    form = los.tunggu_popup(r'Kategori Dokumen', 30)
    los.buffering()
    pilihan = [
        los.pilih_dari_daftar(los.satu(f".//input[@placeholder='{nama}']", 15, form), DATA[kunci])
        for nama, kunci in (('Kategori Dokumen', 'other_document_kategori'),
                            ('Group Dokumen', 'other_document_group'),
                            ('Jenis Dokumen', 'other_document_jenis'))
    ]
    log(f'  other document: {" / ".join(pilihan)}')

    form.find_element(By.XPATH, ".//input[@type='file']").send_keys(BERKAS_PDF)
    los.buffering()
    # Sebagian versi memunculkan popup "berhasil" sendiri setelah berkas dipilih.
    los.konfirmasi_bila_ada(r'berhasil', 'Close', 3)

    nomor = nomor_dokumen_acak()
    los.ketik(los.satu(".//input[@placeholder='Input Nomor Dokumen']", 15, form), nomor)
    los.ketik_tanggal(los.satu(".//input[@placeholder='Input Tanggal Dokumen']", 15, form), hari_ini(),
                      "//*[normalize-space(text())='Upload By']")
    if DATA.get('other_document_keterangan'):
        los.ketik(los.satu(".//textarea[@placeholder='Input Keterangan']", 15, form), DATA['other_document_keterangan'])

    los.klik_xpath(los.tombol('Save', form), 30, induk=form,
                   pesan='Save Add Dokumen belum aktif — ada field wajib yang kosong?')
    los.sukses_lalu_tutup()
    los.satu(f"{xp_section}//tbody/tr[contains(normalize-space(), '{nomor}')]", 60,
             pesan=f'dokumen bernomor {nomor} tidak muncul di tabel Other Document')


def kirim_pembahasan(los: Los, tombol_bawah: str, tombol_konfirmasi: str):
    """
    Tombol bawah (Submit/Approve) → dialog konfirmasi → tombol konfirmasi
    (Submit untuk Staff, Send untuk TL, "Approve & Kirim ke LPS" untuk Kadiv) → "Aksi berhasil".
    """
    los.klik_xpath(f"(//button[normalize-space()={xp_literal(tombol_bawah)}])[last()]", 60,
                   pesan=f'tombol {tombol_bawah} tidak aktif — belum ada dokumen Draft/menunggu?')
    dialog = los.tunggu(
        lambda: next((p for p in los.d.find_elements(By.CSS_SELECTOR, '.MuiDialog-root')
                      if p.is_displayed() and los.cari(los.tombol(tombol_konfirmasi, p), p)), None),
        60, f'dialog dengan tombol "{tombol_konfirmasi}" tidak muncul')
    los.klik_xpath(los.tombol(tombol_konfirmasi, dialog), 15, induk=dialog)
    log(f'  {los.sukses_lalu_tutup(r"Aksi berhasil|berhasil", 300)}')


def fase_staff_pembahasan(los: Los):
    customer = baca_jejak()['customer']
    # Origination List → Detail (kolom Action) → stepper Pembahasan Origination
    baris = tunggu_baris_origination(los, customer, 120)
    los.klik(baris.find_element(By.XPATH, './td[2]//button'))
    los.tunggu(lambda: '/debtor-information' in los.d.current_url or 'origination-creation' in los.d.current_url,
               60, 'detail origination tidak terbuka')
    los.buffering()
    buka_stepper_pembahasan(los)
    isi_tanggal_entry(los)
    isi_dokumen_wajib(los)
    for _ in range(int(DATA.get('other_document_jumlah', 0))):
        tambah_other_document(los)
    kirim_pembahasan(los, 'Submit', 'Submit')


def fase_pembahasan(isi_dokumen: bool, tombol_bawah: str, tombol_konfirmasi: str):
    def kerja(los: Los):
        j = baca_jejak()
        los.buka_tugas(j['customer'], pembahasan(j['fas_id']))
        buka_stepper_pembahasan(los)
        if isi_dokumen:
            isi_dokumen_wajib(los)
        kirim_pembahasan(los, tombol_bawah, tombol_konfirmasi)
    return kerja


def fase_cek_completed(los: Los):
    j = baca_jejak()
    xp = f"//tbody/tr[contains(normalize-space(), {xp_literal(j['fas_id'])})]/td[3]"
    akhir = time.time() + 180
    while True:
        los.buka(RUTE_ORIGINATION)
        status = (los.cari(xp) or [None])[0]
        teks = status.text.strip() if status else '(baris tidak ditemukan)'
        if 'completed' in teks.lower():
            log(f'SELESAI: {j["customer"]} / {j["pipeline_id"]} / {j["fas_id"]} berstatus {teks}')
            return
        if time.time() > akhir:
            raise GagalLangkah(f'{j["fas_id"]} belum Completed, status sekarang: {teks}')
        time.sleep(15)


FASE = [
    ('1-10  Staff Bisnis: customer, pipeline, fasilitas syariah', 'staff_bisnis', fase_buat_pipeline),
    ('11    Staff Bisnis: Submit pipeline', 'staff_bisnis', fase_submit_pipeline),
    ('13-14 TL Bisnis: approve pipeline', 'tl_bisnis', fase_tl_approve_pipeline),
    ('16-17 Staff Bisnis: Go To Origination', 'staff_bisnis', fase_go_to_origination),
    ('18    Staff Bisnis: isi Pembahasan Origination + Other Document, Submit', 'staff_bisnis', fase_staff_pembahasan),
    ('20    TL Bisnis: Submit, Send', 'tl_bisnis', fase_pembahasan(False, 'Submit', 'Send')),
    ('21    Kadiv Bisnis: Approve', 'kadiv_bisnis', fase_pembahasan(False, 'Approve', 'Approve & Kirim ke LPS')),
    ('23    TL DEPI: isi dokumen, Submit', 'tl_depi', fase_pembahasan(True, 'Submit', 'Send')),
    ('25    Kadiv DEPI: Approve', 'kadiv_depi', fase_pembahasan(False, 'Approve', 'Approve & Kirim ke LPS')),
    ('27    TL DH: isi dokumen, Submit', 'tl_dh', fase_pembahasan(True, 'Submit', 'Send')),
    ('29    Kadiv DH: Approve', 'kadiv_dh', fase_pembahasan(False, 'Approve', 'Approve & Kirim ke LPS')),
    ('30    Cek status Completed', 'staff_bisnis', fase_cek_completed),
]

## 9. Jalankan

Run baru menghapus jejak lama. Bila gagal, screenshot halaman terakhir ditampilkan di bawah sel dan
disimpan bersama HTML-nya di folder `artefak/`. Setelah penyebabnya beres, centang `LANJUT` di sel
Konfigurasi, jalankan sel itu, lalu jalankan sel ini lagi: fase yang sudah selesai dilewati dan
customer/pipeline/origination yang sama dipakai ulang.

In [ ]:
from IPython.display import Image, display


def jalankan(lanjut: bool = False, sampai: int = len(FASE)) -> bool:
    if not lanjut and BERKAS_JEJAK.exists():
        BERKAS_JEJAK.unlink()
    log(f'Skenario pembiayaan: Syariah (jejak: {BERKAS_JEJAK.name})')

    los = Los(BASE_URL)
    mulai = time.time()
    try:
        for no, (nama, peran, kerja) in enumerate(FASE[:sampai], start=1):
            if nama in baca_jejak().get('fase_selesai', []):
                log(f'--- lewati (sudah selesai): {nama}')
                continue
            log(f'=== [{no}/{len(FASE)}] {nama} ===')
            los.login(AKUN[peran])
            kerja(los)
            los.logout()
            simpan_jejak(fase_selesai=baca_jejak().get('fase_selesai', []) + [nama])
        durasi = int(time.time() - mulai)
        log(f'Total waktu: {durasi // 60} menit {durasi % 60} detik')
        log(f'Jejak: {json.dumps({k: v for k, v in baca_jejak().items() if k != "fase_selesai"}, ensure_ascii=False)}')
        return True
    except Exception as e:  # noqa: BLE001 — semua kegagalan dilaporkan dengan bukti yang sama
        bukti = los.simpan_bukti('gagal')
        log(f'GAGAL: {e}')
        log(f'Bukti: {bukti}')
        log('Perbaiki penyebabnya, centang LANJUT di sel Konfigurasi, lalu jalankan ulang sel ini.')
        if bukti.exists():
            display(Image(filename=str(bukti)))
        return False
    finally:
        los.tutup()


BERHASIL = jalankan(lanjut=LANJUT, sampai=SAMPAI)

## 10. Jejak run & unduh artefak

In [ ]:
print(json.dumps(baca_jejak(), indent=1, ensure_ascii=False))

if DI_COLAB:
    import shutil as _sh
    from google.colab import files
    arsip = _sh.make_archive('/content/hasil_origination_syariah', 'zip', FOLDER)
    files.download(arsip)